# PolicyFlux — Layer Playground
Eksperymentujemy z kolejnością warstw i nadpisywaniem parametrów.
Celem jest szybkie porównanie wpływu konfiguracji na wyniki głosowań.

In [ ]:
from policyflux.integration import IntegrationConfig, LayerConfig, build_engine
from policyflux.utils.reports import craft_a_bar

SEED = 20260124
NUM_ACTORS = 80
POLICY_DIM = 4
ITERATIONS = 200


def run_with_layers(layer_names, overrides, title):
    config = IntegrationConfig(
        num_actors=NUM_ACTORS,
        policy_dim=POLICY_DIM,
        iterations=ITERATIONS,
        seed=SEED,
        description=title,
        layer_config=LayerConfig(
            layer_names=layer_names,
            layer_overrides=overrides,
        ),
        aggregation_strategy="sequential",
    )

    engine = build_engine(config)
    engine.run_simulation()

    total = len(engine.congress_model.congressmen)
    avg_votes_for = sum(engine.results) / len(engine.results)
    pass_rate = sum(1 for v in engine.results if v > total / 2) / len(engine.results)

    print(engine)
    print(f"Pass rate: {pass_rate:.1%}")

    return {
        "title": title,
        "avg_votes_for": avg_votes_for,
        "avg_votes_against": total - avg_votes_for,
        "pass_rate": pass_rate,
    }

In [ ]:
baseline = run_with_layers(
    layer_names=["ideal_point", "public_opinion", "lobbying", "party_discipline"],
    overrides={
        "public_opinion": {"support_level": 0.55},
        "lobbying": {"intensity": 0.12},
    },
    title="Układ bazowy",
)

media_first = run_with_layers(
    layer_names=["media_pressure", "public_opinion", "ideal_point", "party_discipline"],
    overrides={
        "media_pressure": {"pressure": 0.55},
        "public_opinion": {"support_level": 0.62},
    },
    title="Media na początku",
)

results = [baseline, media_first]
results

In [ ]:
craft_a_bar(
    data=[r["pass_rate"] * 100 for r in results],
    labels=[r["title"] for r in results],
    title="Wpływ kolejności warstw",
    xlabel="Konfiguracja",
    ylabel="Pass rate [%]",
)